# A-001 Beginner Data Exploration + Machine Learning Lab
## From raw sensor data → exploration → Normal / Abnormal classification

This notebook is designed for **complete beginners**.

We will use one enterprise scenario throughout:

> **Asset A-001 — a synthetic cooling-water pump**

The goal is not to build the most advanced model.  
The goal is to understand the **complete data + machine-learning workflow**.

### What you will practice

1. Connect to the SQLite database
2. Load A-001 sensor data
3. Understand rows, columns and data types
4. Check missing values and class balance
5. Perform **univariate analysis**
6. Perform **bivariate analysis**
7. Perform a small **multivariate exploration**
8. Choose feature `X` and target `y`
9. Split data into train/test
10. Train a simple Decision Tree classifier
11. Evaluate the model
12. Use the model on new A-001 readings
13. Try your own experiments

### Important boundary

This notebook **does not predict pump failure**.

It predicts the existing synthetic label:

- `0` = Normal
- `1` = Abnormal

An abnormal prediction means:

> “This reading looks like historical readings labelled abnormal.”

It does **not** mean the pump has failed or that an operational action should be taken automatically.

# PART 1 — Setup

We start with only a few libraries:

- `sqlite3` → read the SQLite database
- `pandas` → work with tables
- `matplotlib` → charts
- `scikit-learn` → machine learning

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Libraries loaded successfully.")

## Database location

This uses the same final Databricks Volume location as the rest of the ENEC training:

`/Volumes/workspace/nuclear_enterprise_360/training_files/nuclear_enterprise_360_v2_2_clean.db`

In [ ]:
DB_PATH = "/Volumes/workspace/nuclear_enterprise_360/training_files/nuclear_enterprise_360_v2_2_clean.db"

print("Database path:")
print(DB_PATH)

# PART 2 — Load A-001 data

For exploration we load several sensor columns.

**Important:** loading many columns for exploration does **not** mean we must use all of them in the model.

In [ ]:
conn = sqlite3.connect(DB_PATH)

query = '''
SELECT
    reading_timestamp,
    asset_id,
    temperature_c,
    vibration_mm_s,
    pressure_bar,
    flow_rate_m3_h,
    electrical_current_a,
    ambient_temperature_c,
    sensor_quality,
    anomaly_flag
FROM sensor_readings
WHERE asset_id = 'A-001'
ORDER BY reading_timestamp
'''

df = pd.read_sql_query(query, conn)
conn.close()

df["reading_timestamp"] = pd.to_datetime(df["reading_timestamp"])

print("Rows loaded:", len(df))
print("Columns loaded:", len(df.columns))
display(df.head(10))

## First beginner question: what is a row and what is a column?

- **One row** = one A-001 sensor reading at one timestamp
- **One column** = one piece of information about that reading

Examples:

- `vibration_mm_s`
- `temperature_c`
- `pressure_bar`
- `anomaly_flag`

In [ ]:
print("Column names:")
for column in df.columns:
    print("-", column)

# PART 3 — Basic data understanding

Before making charts or models, always inspect the dataset.

We want to know:

- How many rows?
- What data types?
- Are values missing?
- What is the numerical range?
- How many Normal vs Abnormal labels?

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
print("Data types:")
print(df.dtypes)

In [ ]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_values"))

In [ ]:
numeric_columns = [
    "temperature_c",
    "vibration_mm_s",
    "pressure_bar",
    "flow_rate_m3_h",
    "electrical_current_a",
    "ambient_temperature_c",
]

print("Numerical summary:")
display(df[numeric_columns].describe().T)

## Target balance: Normal vs Abnormal

This is very important.

If one class is much more common than another, we call it **class imbalance**.

In [ ]:
class_counts = df["anomaly_flag"].value_counts().sort_index()

print("0 = Normal")
print("1 = Abnormal")
print()
print(class_counts)

normal_count = int(class_counts.get(0, 0))
abnormal_count = int(class_counts.get(1, 0))
abnormal_pct = abnormal_count / len(df) * 100

print()
print(f"Normal readings:   {normal_count:,}")
print(f"Abnormal readings: {abnormal_count:,}")
print(f"Abnormal %:        {abnormal_pct:.2f}%")

In [ ]:
plt.figure(figsize=(6, 4))
class_counts.rename(index={0: "Normal", 1: "Abnormal"}).plot(kind="bar")
plt.title("A-001: Normal vs Abnormal Readings")
plt.xlabel("Class")
plt.ylabel("Number of Readings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# PART 4 — Univariate Analysis

## What is univariate analysis?

**Uni = one**

We study **one variable at a time**.

Typical questions:

- What is the average?
- What is the minimum and maximum?
- Is the distribution skewed?
- Are there unusual values?
- How does the variable change over time?

## 4.1 Vibration — summary statistics

In [ ]:
vibration = df["vibration_mm_s"]

print("A-001 vibration summary")
print("-----------------------")
print(f"Count:  {vibration.count():,}")
print(f"Mean:   {vibration.mean():.3f} mm/s")
print(f"Median: {vibration.median():.3f} mm/s")
print(f"Min:    {vibration.min():.3f} mm/s")
print(f"Max:    {vibration.max():.3f} mm/s")
print(f"Std:    {vibration.std():.3f} mm/s")

## 4.2 Vibration distribution — histogram

A histogram answers:

> “Which vibration values occur most often?”

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["vibration_mm_s"], bins=30)
plt.xlabel("Vibration (mm/s)")
plt.ylabel("Number of Readings")
plt.title("A-001 Vibration Distribution")
plt.tight_layout()
plt.show()

## 4.3 Vibration over time

A histogram tells us **what values exist**.

A time plot tells us **when they happened**.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["reading_timestamp"], df["vibration_mm_s"])
plt.xlabel("Time")
plt.ylabel("Vibration (mm/s)")
plt.title("A-001 Vibration Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4.4 Temperature distribution

Try the same univariate thinking with another sensor.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["temperature_c"], bins=30)
plt.xlabel("Temperature (°C)")
plt.ylabel("Number of Readings")
plt.title("A-001 Temperature Distribution")
plt.tight_layout()
plt.show()

## 4.5 Electrical current distribution

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["electrical_current_a"], bins=30)
plt.xlabel("Electrical Current (A)")
plt.ylabel("Number of Readings")
plt.title("A-001 Electrical Current Distribution")
plt.tight_layout()
plt.show()

## 4.6 Sensor quality

Not every useful variable has to be continuous.

`sensor_quality` is categorical.

In [ ]:
print("Sensor quality counts:")
display(df["sensor_quality"].value_counts(dropna=False).to_frame("count"))

plt.figure(figsize=(7, 4))
df["sensor_quality"].value_counts(dropna=False).plot(kind="bar")
plt.xlabel("Sensor Quality")
plt.ylabel("Number of Readings")
plt.title("A-001 Sensor Quality")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# PART 5 — Bivariate Analysis

## What is bivariate analysis?

**Bi = two**

We study **two variables together**.

Examples:

- Vibration vs anomaly
- Vibration vs temperature
- Vibration vs electrical current
- Sensor quality vs anomaly

This helps us see whether variables appear related.

## 5.1 Vibration vs Anomaly

This is the most important bivariate analysis for our simple model.

We want to ask:

> “Are vibration values different for Normal and Abnormal readings?”

In [ ]:
normal_vibration = df.loc[df["anomaly_flag"] == 0, "vibration_mm_s"]
abnormal_vibration = df.loc[df["anomaly_flag"] == 1, "vibration_mm_s"]

print("Normal vibration:")
print(f"  Mean: {normal_vibration.mean():.3f}")
print(f"  Max:  {normal_vibration.max():.3f}")
print()

print("Abnormal vibration:")
print(f"  Mean: {abnormal_vibration.mean():.3f}")
print(f"  Min:  {abnormal_vibration.min():.3f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [normal_vibration, abnormal_vibration],
    tick_labels=["Normal", "Abnormal"]
)
plt.ylabel("Vibration (mm/s)")
plt.title("A-001 Vibration: Normal vs Abnormal")
plt.tight_layout()
plt.show()

### What should you notice?

If the two groups are clearly separated, vibration may be a useful feature for classification.

Do **not** assume this will always happen in real-world data.  
Real sensor data is usually noisier and classes often overlap.

## 5.2 Vibration vs Temperature

Each point below represents one sensor reading.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["vibration_mm_s"],
    df["temperature_c"],
    alpha=0.6
)
plt.xlabel("Vibration (mm/s)")
plt.ylabel("Temperature (°C)")
plt.title("A-001: Vibration vs Temperature")
plt.tight_layout()
plt.show()

corr_vib_temp = df["vibration_mm_s"].corr(df["temperature_c"])
print(f"Correlation between vibration and temperature: {corr_vib_temp:.3f}")

## 5.3 Vibration vs Electrical Current

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["vibration_mm_s"],
    df["electrical_current_a"],
    alpha=0.6
)
plt.xlabel("Vibration (mm/s)")
plt.ylabel("Electrical Current (A)")
plt.title("A-001: Vibration vs Electrical Current")
plt.tight_layout()
plt.show()

corr_vib_current = df["vibration_mm_s"].corr(df["electrical_current_a"])
print(f"Correlation between vibration and current: {corr_vib_current:.3f}")

## 5.4 Compare average sensors by Normal / Abnormal class

This is another easy bivariate technique:

> Group by the target, then compare averages.

In [ ]:
group_summary = (
    df.groupby("anomaly_flag")[numeric_columns]
      .mean()
      .rename(index={0: "Normal", 1: "Abnormal"})
)

display(group_summary.round(3))

## 5.5 Sensor Quality vs Anomaly

This compares **two categorical variables**.

In [ ]:
quality_vs_anomaly = pd.crosstab(
    df["sensor_quality"],
    df["anomaly_flag"],
    margins=True
)

quality_vs_anomaly = quality_vs_anomaly.rename(columns={0: "Normal", 1: "Abnormal"})

display(quality_vs_anomaly)

# PART 6 — Small Multivariate Exploration

## What is multivariate analysis?

We look at **several variables together**.

For beginners, a correlation matrix is a simple starting point.

### Important

Correlation tells us whether numerical variables move together.

It does **not** prove that one variable causes another.

In [ ]:
correlation_columns = [
    "temperature_c",
    "vibration_mm_s",
    "pressure_bar",
    "flow_rate_m3_h",
    "electrical_current_a",
    "ambient_temperature_c",
    "anomaly_flag",
]

corr = df[correlation_columns].corr()

display(corr.round(3))

In [ ]:
plt.figure(figsize=(9, 7))

plt.imshow(corr.values, aspect="auto")
plt.colorbar(label="Correlation")

labels = [
    "Temp",
    "Vibration",
    "Pressure",
    "Flow",
    "Current",
    "Ambient",
    "Anomaly"
]

plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
plt.title("A-001 Correlation Matrix")
plt.tight_layout()
plt.show()

## Exploration checkpoint

Before ML, discuss:

1. Which variable seems most clearly related to `anomaly_flag`?
2. Which variables seem strongly related to each other?
3. Are any variables surprisingly weak?
4. Would adding more variables automatically make the model better?
5. Could highly related variables contain duplicate information?

For our **first ML model**, we will still use only:

> `vibration_mm_s → anomaly_flag`

The exploration can be broad even when the model stays simple.

# PART 7 — Prepare the Machine Learning Problem

In machine learning:

- **X = features / inputs**
- **y = target / answer**

For the beginner model:

- `X = vibration_mm_s`
- `y = anomaly_flag`

In [ ]:
X = df[["vibration_mm_s"]]
y = df["anomaly_flag"]

print("Feature X:")
display(X.head())

print("Target y:")
display(y.head())

print()
print("X shape:", X.shape)
print("y shape:", y.shape)

# PART 8 — Train / Test Split

We do not want to test the model using the same examples it studied.

We use:

- **75% training data**
- **25% testing data**

`stratify=y` helps preserve the rare Normal/Abnormal balance in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print()

print("Training class counts:")
print(y_train.value_counts().sort_index())
print()

print("Testing class counts:")
print(y_test.value_counts().sort_index())

# PART 9 — Train a Simple Decision Tree

We deliberately use:

`max_depth=1`

That means the tree can make only **one decision**.

This makes the model very easy to understand.

In [ ]:
model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42,
)

model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

# PART 10 — Predict on Unseen Test Data

In [ ]:
predictions = model.predict(X_test)

results = X_test.copy()
results["Actual"] = y_test.values
results["Predicted"] = predictions

display(results.head(15))

# PART 11 — Evaluate the Model

For classification we should not look only at accuracy.

We will calculate:

- **Accuracy** → overall percentage correct
- **Precision** → when we say Abnormal, how often are we right?
- **Recall** → of all actual Abnormal readings, how many did we catch?
- **F1** → balance between precision and recall
- **Balanced Accuracy** → gives both classes more equal importance
- **Confusion Matrix** → shows the four kinds of classification result

In [ ]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
balanced_acc = balanced_accuracy_score(y_test, predictions)
cm = confusion_matrix(y_test, predictions)

print(f"Accuracy:          {accuracy * 100:.2f}%")
print(f"Precision:         {precision * 100:.2f}%")
print(f"Recall:            {recall * 100:.2f}%")
print(f"F1 Score:          {f1 * 100:.2f}%")
print(f"Balanced Accuracy: {balanced_acc * 100:.2f}%")
print()
print("Confusion Matrix:")
print(cm)

## Confusion Matrix — visual version

Matrix layout:

|              | Predicted Normal | Predicted Abnormal |
|--------------|------------------|--------------------|
| Actual Normal | Correct Normal | False Alarm |
| Actual Abnormal | Missed Abnormal | Correct Abnormal |

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)

plt.title("A-001 Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks([0, 1], ["Normal", "Abnormal"])
plt.yticks([0, 1], ["Normal", "Abnormal"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)

plt.colorbar()
plt.tight_layout()
plt.show()

## Important discussion: why can accuracy be misleading?

Our A-001 dataset contains many more Normal readings than Abnormal readings.

A model that predicts **Normal almost every time** could still look very accurate.

That is why rare-event problems should also use:

- Recall
- Precision
- Confusion matrix
- Balanced metrics

# PART 12 — Inspect the Rule the Model Learned

Because the tree has only one split, we can see the threshold directly.

In [ ]:
threshold = model.tree_.threshold[0]

print(f"Learned vibration threshold: approximately {threshold:.3f} mm/s")
print()
print(f"If vibration <= {threshold:.3f} mm/s → NORMAL")
print(f"If vibration >  {threshold:.3f} mm/s → ABNORMAL")

In [ ]:
plt.figure(figsize=(10, 5))
plot_tree(
    model,
    feature_names=["vibration_mm_s"],
    class_names=["Normal", "Abnormal"],
    filled=False,
    rounded=True,
    impurity=False,
)
plt.title("A-001 Beginner Decision Tree")
plt.show()

# PART 13 — Turn the Model into a Tiny A-001 Classifier

This is a simple way to make the notebook feel like an application.

Give it a vibration value and it returns:

- the prediction
- the meaning
- the learned threshold

In [ ]:
def classify_a001(vibration_mm_s):
    new_data = pd.DataFrame({
        "vibration_mm_s": [vibration_mm_s]
    })

    prediction = int(model.predict(new_data)[0])
    meaning = "NORMAL" if prediction == 0 else "ABNORMAL"

    return {
        "asset": "A-001",
        "vibration_mm_s": vibration_mm_s,
        "prediction": prediction,
        "meaning": meaning,
        "learned_threshold_mm_s": round(float(threshold), 3),
    }

classify_a001(4.8)

## Try several new readings

In [ ]:
new_readings = pd.DataFrame({
    "vibration_mm_s": [1.2, 3.0, 4.8, 5.0, 5.2, 5.5]
})

new_readings["prediction"] = model.predict(new_readings)
new_readings["meaning"] = new_readings["prediction"].map({
    0: "NORMAL",
    1: "ABNORMAL",
})

display(new_readings)

# PART 14 — Student Exploration Lab

Now let participants **play with the data**.

The goal is not to copy code.  
The goal is to ask questions and see what happens.

## Exercise A — Pick another variable for univariate analysis

Try one of:

- `pressure_bar`
- `flow_rate_m3_h`
- `ambient_temperature_c`

Questions:

1. What is its average?
2. What is its min/max?
3. What does its histogram look like?
4. Does it contain unusual values?

In [ ]:
# TRY IT YOURSELF
column_to_explore = "pressure_bar"

print(df[column_to_explore].describe())

plt.figure(figsize=(8, 4))
plt.hist(df[column_to_explore], bins=30)
plt.xlabel(column_to_explore)
plt.ylabel("Count")
plt.title(f"Distribution of {column_to_explore}")
plt.tight_layout()
plt.show()

## Exercise B — Try another bivariate relationship

Change `y_column` and see what happens.

In [ ]:
# TRY IT YOURSELF
x_column = "vibration_mm_s"
y_column = "flow_rate_m3_h"

plt.figure(figsize=(8, 5))
plt.scatter(df[x_column], df[y_column], alpha=0.6)
plt.xlabel(x_column)
plt.ylabel(y_column)
plt.title(f"{x_column} vs {y_column}")
plt.tight_layout()
plt.show()

print("Correlation:", round(df[x_column].corr(df[y_column]), 3))

## Exercise C — Train the same simple tree using a DIFFERENT single feature

This is a great experiment.

Try:

- temperature
- current
- pressure
- flow

Then compare the metrics.

### Question

Does a different feature perform as well as vibration?

In [ ]:
# TRY IT YOURSELF
experimental_feature = "temperature_c"

X_exp = df[[experimental_feature]]

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_exp,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

experimental_model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42,
)

experimental_model.fit(X_train_exp, y_train_exp)

experimental_predictions = experimental_model.predict(X_test_exp)

print("Experimental feature:", experimental_feature)
print("Accuracy:", round(accuracy_score(y_test_exp, experimental_predictions) * 100, 2), "%")
print("Recall:", round(recall_score(y_test_exp, experimental_predictions, zero_division=0) * 100, 2), "%")
print("F1:", round(f1_score(y_test_exp, experimental_predictions, zero_division=0) * 100, 2), "%")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_exp, experimental_predictions))

# OPTIONAL — Multi-feature experiment

This is **not required for the beginner project**.

Only try this after the one-feature model is understood.

Here we allow the Decision Tree to look at several sensor variables.

In [ ]:
multi_features = [
    "vibration_mm_s",
    "temperature_c",
    "pressure_bar",
    "flow_rate_m3_h",
    "electrical_current_a",
    "ambient_temperature_c",
]

X_multi = df[multi_features]

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

multi_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42,
)

multi_model.fit(X_train_multi, y_train_multi)

multi_predictions = multi_model.predict(X_test_multi)

print("OPTIONAL multi-feature model")
print("Accuracy:", round(accuracy_score(y_test_multi, multi_predictions) * 100, 2), "%")
print("Recall:", round(recall_score(y_test_multi, multi_predictions, zero_division=0) * 100, 2), "%")
print("F1:", round(f1_score(y_test_multi, multi_predictions, zero_division=0) * 100, 2), "%")

In [ ]:
feature_importance = pd.DataFrame({
    "feature": multi_features,
    "importance": multi_model.feature_importances_,
}).sort_values("importance", ascending=False)

display(feature_importance)

# FINAL TAKEAWAYS

You have now completed much more than “train a model”.

You practiced the real workflow:

**Database → Data → Quality Checks → EDA → Features → Target → Train/Test → Model → Prediction → Evaluation → Interpretation**

### Core beginner project

**A-001 vibration → Decision Tree → Normal / Abnormal**

### What should happen after an Abnormal result?

Not an automatic maintenance action.

Instead:

**Prediction → inspect evidence → review work orders / reports / approved procedures → qualified human decision**

That becomes our bridge into **Generative AI + RAG**.